# 10주차 · Transformer와 BERT
**충북대학교 | 자연언어처리 | 임화정**

## 학습목표
Q·K·V와 scaled dot-product Attention을 계산하고, padding/causal mask를 구분합니다. Multi-head, 위치 정보, 잔차 연결, LayerNorm, FFN을 설명하고 BERT의 자기지도 사전학습과 파인튜닝을 구분합니다.

## 실행 안내
Colab/Jupyter에서 위에서부터 실행하세요. Python 3.10 이상, numpy와 pandas가 필요합니다. 기본 실습은 CPU로 실행하며 모델 다운로드·GPU·인터넷이 필요 없습니다. 라이브러리가 없을 때만 `%pip install numpy pandas`를 별도 셀에서 실행하세요.

**중요:** 모든 입력 벡터와 가중치는 시드 고정 랜덤 교육용 값입니다. 훈련된 BERT의 출력이나 의미 이해를 보여주는 것이 아닙니다. Attention 가중치를 인과적 설명 또는 언어적 관계의 정답으로 해석하지 않습니다.

권장 운영안: 개념45 → 실습35 → 휴식10 → 실습45 → 정리25분, 총160분(공식 시수 아님).

9주차에서 평균 pooling의 어순 손실과 RNN의 순차 계산을 확인했습니다. 이번 주는 토큰 간 직접 정보 교환, 다음11주차는 사전학습 BERT를 실제 분류 과제에 파인튜닝합니다.

**학생용: TODO·확인 질문·응답란 포함**

In [2]:
import sys
import numpy as np
import pandas as pd
from IPython.display import display
SEED = 42
rng = np.random.default_rng(SEED) # 난수 생성기
tokens = ['나는', '오늘', '영화', '본다', '[PAD]']
X = rng.normal(0, 0.3, (5, 8)) # 평균 0, 표편 0.3, (5, 8) 형태의 배열 반환.
Wq = rng.normal(0, 0.3, (8, 4))
Wk = rng.normal(0, 0.3, (8, 4))
Wv = rng.normal(0, 0.3, (8, 4))
print('Python:', sys.version.split()[0], '| numpy:', np.__version__, '| pandas:', pd.__version__)
print('X:', X.shape, '| Q/K/V projections:', Wq.shape)


Python: 3.13.18 | numpy: 2.4.6 | pandas: 2.2.3
X: (5, 8) | Q/K/V projections: (8, 4)


## 1. Stable softmax
각 query 행에서 key 방향으로 합이 1인 가중치를 만듭니다. 행 최대를 빼 overflow를 줄입니다. 모두 마스킹된 행에서는 softmax가 정의되지 않으므로 이 실습은 오류를 명시적으로 발생시킵니다.

### 실습 A
TODO에서 행 최대를 뺀 뒤 지수화하고 행 합으로 나누세요.

In [3]:
def softmax_rows(scores):
    if not np.all(np.isfinite(scores).any(axis=-1)): # 배열의 각 원소에 대해 유한수이면 True를 반환 -> 마지막 축(axis=-1)을 기준으로 True가 하나라도 있으면 True를 반환
        raise ValueError('각 query에는 최소 한 개의 유효 key가 필요합니다.')
    # TODO A: 안정적인 행별 softmax로 변경하세요.
    shifted = scores - np.max(scores, axis=-1, keepdims=True) # 마지막 축(열 번호 바뀌는 방향) 기준으로 계산한 최댓값을 scores의 각 행에서 빼줌
    exp = np.exp(shifted) # e^shifted 계산
    return exp / exp.sum(axis=-1, keepdims=True)

probe = softmax_rows(np.array([[1000., 1001., 1002.]]))
print(probe)
print('완료 확인:', np.allclose(probe, [[0.09003057, 0.24472847, 0.66524096]]))


[[0.09003057 0.24472847 0.66524096]]
완료 확인: True


## 2. Q·K·V → Attention
Q=XWq, K=XWk, V=XWv. 점수는 QKᵀ/√dk, A=softmax(점수), 출력=AV입니다. Q·K는 비교에, V는 가중합할 내용에 사용합니다. 이 예제는 dk=dv=4이지만 반드시 같아야 하는 것은 아닙니다.

### 실습 B
점수 계산을 완성하세요. 행=query, 열=key입니다. 입력을 그대로 가중합하는 것이 아니라 V를 가중합합니다.

In [4]:
def self_attention(x, wq, wk, wv, allowed=None):
    q, k, v = x @ wq, x @ wk, x @ wv # @는 행렬 곱셈 연산자
    # TODO B: 아래 0점수를 QK^T / sqrt(dk)로 변경하세요.
    scores = q @ k.T / np.sqrt(k.shape[-1])
    if allowed is not None:
        scores = np.where(allowed, scores, -np.inf)
    weights = softmax_rows(scores)
    return weights @ v, weights

output, weights = self_attention(X, Wq, Wk, Wv)
print('output:', output.shape, '| weights:', weights.shape)
display(pd.DataFrame(weights, index=tokens, columns=tokens).round(4))
print('완료 확인(점수가 모두 같은 것은 아님):', not np.allclose(weights, weights[0]))


output: (5, 4) | weights: (5, 5)


,나는,오늘,영화,본다,[PAD]
나는,0.1947,0.2055,0.2010,0.2077,0.1912
오늘,0.2032,0.1972,0.2007,0.2006,0.1983
영화,0.1983,0.2020,0.1983,0.1962,0.2052
본다,0.2000,0.1999,0.1985,0.1975,0.2041
[PAD],0.1966,0.2060,0.1988,0.2055,0.1931


완료 확인(점수가 모두 같은 것은 아님): True


## 3. Padding mask와 causal mask
Padding mask는 실제 문장이 아닌 key 위치를 제거합니다. Causal mask는 미래 key를 제거하며 생성형 decoder에서 사용됩니다. BERT encoder는 일반적으로 미래 위치를 막지 않습니다.
- 양방향이기 때문입니다?

**padding query:** key를 막아도 PAD 행 출력은 자동으로0이 되지 않습니다. 여기서는 PAD query 출력을 별도로0으로 설정합니다. 모든 행을 차단하면 softmax가 정의되지 않으므로 query 제거를 이후에 적용합니다.

### 실습 C
아래쪽 삼각형(대각 포함)의 causal 허용 행렬을 만드세요.

In [5]:
key_valid = np.array([True, True, True, True, False])
padding_allowed = np.broadcast_to(key_valid, (5, 5))
# TODO C: 대각 포함 아래쪽 삼각형 bool 행렬로 바꾸세요.
causal_allowed = np.tril(np.ones((5, 5), dtype=bool)) # 대각선 위쪽 요소를 0으로 만든 하삼각행렬 반환
combined_allowed = padding_allowed & causal_allowed
pad_output, pad_weights = self_attention(X, Wq, Wk, Wv, padding_allowed)
causal_output, causal_weights = self_attention(X, Wq, Wk, Wv, combined_allowed)
pad_output[~key_valid] = 0
causal_output[~key_valid] = 0
display(pd.DataFrame(causal_weights, index=tokens, columns=tokens).round(4))
print('미래 Attention 차단 확인:', np.allclose(causal_weights[np.triu_indices(5, k=1)], 0))


,나는,오늘,영화,본다,[PAD]
나는,1.0000,0.0000,0.0000,0.0000,0.0
오늘,0.5074,0.4926,0.0000,0.0000,0.0
영화,0.3313,0.3374,0.3313,0.0000,0.0
본다,0.2513,0.2512,0.2493,0.2482,0.0
[PAD],0.2437,0.2553,0.2464,0.2546,0.0


미래 Attention 차단 확인: True


## 4. Multi-head와 출력 투영
두 head가 서로 다른 QKV 가중치를 사용합니다. 출력(5×4)을 두 개 연결하면(5×8), Wo(8×8)로 투영합니다. head마다 언어 기능이 미리 지정되는 것은 아닙니다. 아래 두 head는 랜덤이며 문법 head/의미 head라고 부를 수 없습니다.

In [6]:
head_outputs, head_weights = [], []
for head_index in range(2):
    matrices = [rng.normal(0, 0.3, (8, 4)) for _ in range(3)]
    head, attn = self_attention(X, *matrices, allowed=padding_allowed)
    head_outputs.append(head)
    head_weights.append(attn)
Wo = rng.normal(0, 0.3, (8, 8))
concatenated = np.concatenate(head_outputs, axis=-1)
multihead = concatenated @ Wo
print('concat:', concatenated.shape, '| output:', multihead.shape) # 평균이 아니라 선형으로 재조합한 것입니다?
assert multihead.shape == X.shape


concat: (5, 8) | output: (5, 8)


## 5. 위치 없는 Attention의 순열 등변성
입력 순서를 P로 바꾸면 출력도 같은 P로 바뀝니다. 따라서 이는 출력이 그대로인 **불변성**이 아니라 **등변성**입니다. 아래는 PAD를 제외한 네 토큰을 비교하며 mask는 적용하지 않습니다.

원래 Transformer의 사인·코사인 위치 인코딩을 추가한 뒤 토큰만 섞고 위치는 고정하면 일반적으로 그 성질이 깨집니다. BERT는 원래 학습형 위치 임베딩을 사용하므로 사인 방식과 동일하지 않습니다.

### 실습 D
짝수 차원은 sin, 홀수 차원은 cos인 위치 인코딩을 완성하세요.

In [8]:
def sinusoidal_positions(length, dim):
    positions = np.arange(length)[:, None]
    frequencies = 10000 ** (-np.arange(0, dim, 2) / dim)
    angles = positions * frequencies
    pe = np.zeros((length, dim))
    # TODO D: pe[:,0::2]에 sin, pe[:,1::2]에 cos 값을 할당하세요.
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles)
    return pe

base = X[:4]
perm = np.array([2, 0, 3, 1])
y, _ = self_attention(base, Wq, Wk, Wv)
yp, _ = self_attention(base[perm], Wq, Wk, Wv)
print('위치 없음: 등변성', np.allclose(yp, y[perm])) # True?
pe = sinusoidal_positions(4, 8)
ypos, _ = self_attention(base + pe, Wq, Wk, Wv)
ypos_permuted, _ = self_attention(base[perm] + pe, Wq, Wk, Wv)
print('토큰만 섞고 위치 고정: 등변성', np.allclose(ypos_permuted, ypos[perm])) # False?
display(pd.DataFrame(pe).round(4))
print('완료 확인:', not np.allclose(pe, 0))


위치 없음: 등변성 True
토큰만 섞고 위치 고정: 등변성 False


,0,1,2,3,4,5,6,7
0,0.0000,1.0000,0.0000,1.0000,0.00,1.0000,0.000,1.0
1,0.8415,0.5403,0.0998,0.9950,0.01,1.0000,0.001,1.0
2,0.9093,-0.4161,0.1987,0.9801,0.02,0.9998,0.002,1.0
3,0.1411,-0.9900,0.2955,0.9553,0.03,0.9996,0.003,1.0


완료 확인: True


## 6. 잔차 연결·LayerNorm·FFN
Transformer에는 Attention 외에도 위치별 비선형 FFN이 있습니다. LayerNorm은 각 토큰의 feature 축을 정규화하며 문장/배치 전체 정규화와 다릅니다. 아래는 원래 post-LN 구조의 개념 예시이고 dropout을 생략합니다. gamma=1,beta=0으로 고정했으며 실제 모델은 학습합니다. 다른 모델의 pre-LN 변형과 구분하세요.

In [9]:
def layer_norm(x, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    variance = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(variance + eps)

z = layer_norm(X + multihead)
W1 = rng.normal(0, 0.3, (8, 16))
W2 = rng.normal(0, 0.3, (16, 8))
ffn = np.maximum(0, z @ W1) @ W2  # 교육용 ReLU FFN, bias=0
block_output = layer_norm(z + ffn)
block_output[~key_valid] = 0
print('블록 출력:', block_output.shape)
assert np.allclose(block_output[:4].mean(axis=-1), 0, atol=1e-8)


블록 출력: (5, 8)


## 7. Encoder와 decoder 비교
- Encoder: padding을 제외한 좌우 문맥을 사용할 수 있습니다. -> 문맥 이해에 적합합니다?
- 원래 encoder-decoder의 decoder: causal self-attention과 encoder 출력(k/v)에 대한 cross-attention을 사용합니다.
- Cross-attention: Q는 decoder, K/V는 encoder에서 오며 두 길이는 달라도 됩니다.
- BERT: encoder 기반. GPT 계열: decoder-only 기반(원래 번역 decoder의 cross-attention이 그대로 필수인 것은 아님).
- 병렬화: 훈련 중 토큰 계산을 병렬화할 수 있지만 일반적인 자기회귀 생성은 새 토큰을 순차 생성합니다.
- 기본 dense attention의 점수 행렬은 길이 n에 대해 n×n입니다. 긴 입력에는 시간·메모리 부담이 있습니다.


## 8. BERT 입력·MLM·NSP 목표 만들기
원래 BERT 입력은 token + position + segment 임베딩의 합입니다. `[CLS] A [SEP] B [SEP]`가 문장쌍 예시입니다. WordPiece는 학습된 subword 토크나이저이며 FastText 문자 n-gram과 다릅니다.

아래 입력은 **손으로 만든 토큰 예시**입니다. 실제 BERT tokenizer/ID/추론이 아닙니다. MLM 대상만 label에 남기고 나머지는 -100(일반적인 손실 ignore index 표기)을 사용합니다. 원래 BERT는 약15% 토큰을 대상으로 선택한 뒤 그중80% MASK,10% 랜덤 토큰,10% 유지합니다. 다음 셀은 원리 설명을 위해 한 위치를 지정하여 MASK로 바꿉니다. 80/10/10 비율을 짧은 한 문장에서 재현하는 코드가 아닙니다.

In [10]:
bert_tokens = ['[CLS]', '나는', '오늘', '영화', '본다', '[SEP]']
selected_index = 3
mlm_input = bert_tokens.copy()
mlm_labels = [-100] * len(bert_tokens)
mlm_labels[selected_index] = bert_tokens[selected_index]
mlm_input[selected_index] = '[MASK]'
display(pd.DataFrame({'original': bert_tokens, 'mlm_input': mlm_input, 'target': mlm_labels}))
assert mlm_labels[3] == '영화' and mlm_input[3] == '[MASK]'
# 원문 연속 여부가 알려진 자체작성 교육용 문장쌍
nsp_examples = pd.DataFrame([
    {'A': '나는 영화관에 갔다.', 'B': '표를 사고 자리에 앉았다.', 'IsNext': 1}, # 관련O?
    {'A': '나는 영화관에 갔다.', 'B': '사과는 달콤한 과일이다.', 'IsNext': 0} # 관련X?
])
display(nsp_examples)
print('NSP label은 이 실습의 자체 규약: IsNext=1, NotNext=0 (라이브러리 ID와 혼동 금지)')


,original,mlm_input,target
0,[CLS],[CLS],-100
1,나는,나는,-100
2,오늘,오늘,-100
3,영화,[MASK],영화
4,본다,본다,-100
5,[SEP],[SEP],-100


,A,B,IsNext
0,나는 영화관에 갔다.,표를 사고 자리에 앉았다.,1
1,나는 영화관에 갔다.,사과는 달콤한 과일이다.,0


NSP label은 이 실습의 자체 규약: IsNext=1, NotNext=0 (라이브러리 ID와 혼동 금지)


### 해석 질문
MLM은 감성 라벨 없이 무엇을 정답으로 쓰나요? NSP는 모든 BERT 후속 모델의 필수 과제인가요? `[CLS]`에 분류층을 붙였다고 바로 감성 분석기가 되나요?


- 정답은 선택한 원래 토큰입니다. 자가지도학습은 정답이 전혀 없는 학습이 아니라 원문에서 목표를 생성합니다.
- NSP는 원래 BERT의 과제이며, 모든 후속 모델에서 사용하는 것은 아닙니다.
- 랜덤 분류층은 감성 과제를 아직 학습하지 않았으므로, 라벨 데이터로 파인튜닝하거나 이미 해당 과제로 학습된 체크포인트가 필요합니다. [CLS]는 문장 분류에 자주 쓰지만 최적 의미 벡터임이 자동 보장되지 않습니다.

## 9. 사전학습·추론·파인튜닝
사전학습은 대규모 텍스트에서 일반 표현을 학습합니다. 추론은 저장된 가중치로 예측만 수행하고, 파인튜닝은 과제 손실에 따라 가중치를 갱신합니다. 다음 주는 실제 모델/토크나이저 다운로드와 라벨 데이터의 train/validation/test 분할을 수행합니다.

- 라벨 규약, 토크나이저, 최대길이와 잘림 정책을 함께 기록합니다.
- train으로 학습하고 validation으로 선택, test는 최종1회 평가합니다.
- 같은 문장·작성자·상품 등의 중복 및 그룹 누수를 점검합니다.
- 큰 사전학습 모델이 항상 TF-IDF baseline보다 우수한 것은 아닙니다. 공정한 분할/지표/자원 조건으로 비교합니다.


## 10. 과제와 응답란
1. 실습 A/B/C/D TODO를 완성하세요.
2. Q,K,V,점수,출력의 shape를 써주세요.
3. padding 열과 미래 위치가0인지 확인하고 둘의 용도를 설명하세요.
4. 위치 정보 없는 attention이 불변이 아닌 등변인 이유를 쓰세요.
5. multi-head가 단일 head 여러 개의 평균과 다른 이유를 설명하세요.
6. 원래 BERT와 원래 Transformer의 위치 정보 차이를 쓰세요.
7. MLM 정답과 NSP 정답의 생성 방식을 설명하세요.
- MLM과 달리, NSP는 문장 간의 연결을 고려합니다.
8. 랜덤 가중치 실습으로 문장 의미를 이해했다고 주장할 수 없는 이유를 쓰세요.

**선택 확장:** Wq/Wk 크기를10배로 바꾸고 attention 분포가 어떻게 변하는지 관찰하세요. 점수의 집중과 의미 품질을 혼동하지 말고 √dk 스케일의 필요성을 설명하세요.

### 응답란
- 차원: (5, 4)
- mask 구분: padding mask는 실제 문장이 아닌 key 위치를 제거합니다. Causal mask는 미래 key를 제거하며 생성형 decoder에서 사용됩니다.
- 위치/순열 실험: 입력 순서를 바꾸면 출력도 이와 동일하게 바뀌기 때문입니다.
- multi-head 해석: 단순 평균은 각 헤드의 차원별 위치 정보를 섞어버리기 때문입니다.
- BERT 사전학습 목표: BERT는 Transformer와 달리 학습형 위치 임베딩을 사용합니다.
- 한계/11주차 계획:

### 제출 체크리스트
- [ ] TODO 완료 후 처음부터 순서대로 실행, 결과 저장
- [ ] 시드·버전·랜덤 가중치 개념 데모 명시
- [ ] attention 행/열 방향과 행합 확인
- [ ] 의미 정확도나 분류 성능을 측정했다고 주장하지 않음

제출 예시: `학번_이름_10주차.ipynb`와 해석 README. 공식 제출일/방식/배점은 담당 교원 안내를 따릅니다.

## 참고 자료
- 제공 「Part3 딥러닝 9–12주차 강의안」 10주차(10–14쪽)
- 제공 「06 텍스트 분류」 디자인 및 강의계획
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [BERT](https://arxiv.org/abs/1810.04805)

원자료 보완: LayerNorm 용어, Transformer의 FFN·정규화 요소, BERT 학습형 위치 임베딩, NSP의 적용 범위, 훈련 병렬화와 생성 순차 처리의 차이를 명확히 했습니다.